# Wide & Deep — Wide & Deep Learning for Recommender Systems (2016)

_Papers · Recommender Systems_

**Train a memorizing linear model and a generalizing deep network together, summed before one sigmoid.**

---

This notebook is a practice scaffold for the **Wide & Deep — Wide & Deep Learning for Recommender Systems (2016)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch, torch.nn as nn, numpy as np

torch.manual_seed(0); np.random.seed(0)

# --- 0. Worked example: cross-product transformation phi (Eqn. 1), AND(user=u1, app=a2). ---
# phi_k(x) = prod_i x_i^{c_ki}; here c=1 only for "user is u1" and "app is a2".
def cross(user_onehot, app_onehot):
    return user_onehot[1] * app_onehot[2]          # AND of the two selected indicators
print("phi AND(user=u1,app=a2):",
      [cross([0,1,0],[0,0,1]),                      # u1,a2  -> 1
       cross([0,1,0],[0,1,0]),                      # u1,a1  -> 0
       cross([1,0,0],[0,0,1])])                     # u0,a2  -> 0
# phi AND(user=u1,app=a2): [1, 0, 0]

# --- 1. A toy task that needs BOTH memorization and generalization. ---
U, A, EMB = 150, 150, 5
g = torch.Generator().manual_seed(0)
taste = torch.randn(U, EMB, generator=g); appv = torch.randn(A, EMB, generator=g)
smooth = torch.sigmoid((taste @ appv.T) * 1.3)     # smooth, generalizable relevance surface

rng = np.random.RandomState(7)                     # exception pairs sit where smooth is LOW,
low = (smooth < 0.2)                               # so memorizing y=1 CONTRADICTS the smooth surface
cands = [(int(u), int(a)) for u in range(U) for a in range(A) if low[u, a]]
rng.shuffle(cands); exc = cands[:120]; exc_set = set(exc)
def lp(u, a): return 0.97 if (u, a) in exc_set else float(smooth[u, a])

def make(n, seed, force=0.0):                       # 'force' makes exception pairs recur
    r = np.random.RandomState(seed)
    us = r.randint(0, U, n).tolist(); as_ = r.randint(0, A, n).tolist()
    for i in range(int(n * force)):
        u, a = exc[r.randint(len(exc))]; us[i] = u; as_[i] = a
    ys = np.array([1.0 if r.rand() < lp(u, a) else 0.0 for u, a in zip(us, as_)], np.float32)
    return torch.tensor(us), torch.tensor(as_), torch.tensor(ys)

tr_u, tr_a, tr_y = make(14000, 10, force=0.15)
te_u, te_a, te_y = make(5000, 99, force=0.15)

# Wide cross feature: one weight per (user,app) pair SEEN in training; unseen -> OOV bucket.
pair_id = {}
for u, a in zip(tr_u.tolist(), tr_a.tolist()): pair_id.setdefault((u, a), len(pair_id))
NP = len(pair_id)
pix = lambda u, a: pair_id.get((int(u), int(a)), NP)
tr_p = torch.tensor([pix(u, a) for u, a in zip(tr_u, tr_a)])
te_p = torch.tensor([pix(u, a) for u, a in zip(te_u, te_a)])
NPID = NP + 1
print(f"train pairs={NP}  test rows with UNSEEN pair = {(te_p == NP).float().mean():.1%}")

def auc(y, p):                                      # rank-based ROC-AUC
    y = y.numpy(); p = p.detach().numpy()
    o = np.argsort(p); r = np.empty(len(p)); r[o] = np.arange(len(p))
    pos = (y == 1); npos = pos.sum(); nneg = len(y) - npos
    if npos == 0 or nneg == 0: return float('nan')
    return float((r[pos].sum() - npos * (npos - 1) / 2) / (npos * nneg))

# --- 2. The Wide & Deep model. mode in {"wide","deep","both"} for the ablation. ---
class WideDeep(nn.Module):
    def __init__(self, mode):
        super().__init__(); self.mode = mode
        self.wide = nn.Embedding(NPID, 1); nn.init.zeros_(self.wide.weight)   # WIDE: cross weight
        self.eu = nn.Embedding(U, EMB); self.ea = nn.Embedding(A, EMB)        # DEEP: embeddings
        self.mlp = nn.Sequential(nn.Linear(2 * EMB, 16), nn.ReLU(), nn.Linear(16, 1))
        self.b = nn.Parameter(torch.zeros(1))                                 # shared bias b
    def forward(self, u, a, p):
        z = self.b.expand(len(u)).clone()
        if self.mode in ("wide", "both"):
            z = z + self.wide(p).squeeze(1)                                   # w_wide . [x, phi(x)]
        if self.mode in ("deep", "both"):
            z = z + self.mlp(torch.cat([self.eu(u), self.ea(a)], 1)).squeeze(1)  # w_deep . a^(lf)
        return z                                     # Eqn. 3 logit; sigmoid is inside BCEWithLogits

def run(mode, epochs=150, lr=0.03, wd=3e-4):
    torch.manual_seed(0); net = WideDeep(mode)
    opt = torch.optim.Adam(net.parameters(), lr=lr, weight_decay=wd)
    lf = nn.BCEWithLogitsLoss()                      # ONE joint loss -> back-props into BOTH parts
    for _ in range(epochs):
        opt.zero_grad(); lf(net(tr_u, tr_a, tr_p), tr_y).backward(); opt.step()
    p = torch.sigmoid(net(te_u, te_a, te_p))
    is_exc = torch.tensor([1 if (int(u), int(a)) in exc_set else 0 for u, a in zip(te_u, te_a)])
    rec = float((p[is_exc == 1] > 0.5).float().mean())                       # exception recall
    unseen = te_p == NP
    return auc(te_y, p), auc(te_y[unseen], p[unseen]), rec

print(f"\n{'mode':6s}{'test AUC':>10s}{'unseen AUC':>12s}{'exc recall':>12s}")
for m in ["wide", "deep", "both"]:
    a, gen, rec = run(m)
    print(f"{m:6s}{a:10.4f}{gen:12.4f}{rec:12.3f}")
# wide:   strong memorization (exc recall ~1.0) but unseen-pair AUC ~0.50 (chance) -> no generalization.
# deep:   generalizes on unseen pairs but weaker overall.
# both:   highest overall test AUC -> wide&deep beats wide-only and deep-only.
# (Our small run, not the paper's reported numbers; exact values vary by seed/hardware.)

## Visualize the data & results

_On a task needing BOTH a memorized cross-feature rule AND smooth generalization, does wide&deep beat wide-only and deep-only?_

In [ ]:
import torch, torch.nn as nn, numpy as np
# Reproduces the qualitative effect: wide&deep > wide-only and deep-only on a task that
# needs BOTH a memorized cross-feature rule AND smooth generalization. Tiny + CPU.
torch.manual_seed(0); np.random.seed(0)
U, A, EMB = 150, 150, 5
g = torch.Generator().manual_seed(0)
taste = torch.randn(U, EMB, generator=g); appv = torch.randn(A, EMB, generator=g)
smooth = torch.sigmoid((taste @ appv.T) * 1.3)
rng = np.random.RandomState(7)
cands = [(int(u), int(a)) for u in range(U) for a in range(A) if smooth[u, a] < 0.2]
rng.shuffle(cands); exc = cands[:120]; exc_set = set(exc)
lp = lambda u, a: 0.97 if (u, a) in exc_set else float(smooth[u, a])
def make(n, seed, force):
    r = np.random.RandomState(seed)
    us = r.randint(0, U, n).tolist(); as_ = r.randint(0, A, n).tolist()
    for i in range(int(n * force)):
        u, a = exc[r.randint(len(exc))]; us[i] = u; as_[i] = a
    ys = np.array([1.0 if r.rand() < lp(u, a) else 0.0 for u, a in zip(us, as_)], np.float32)
    return torch.tensor(us), torch.tensor(as_), torch.tensor(ys)
tr_u, tr_a, tr_y = make(14000, 10, 0.15); te_u, te_a, te_y = make(5000, 99, 0.15)
pair_id = {}
for u, a in zip(tr_u.tolist(), tr_a.tolist()): pair_id.setdefault((u, a), len(pair_id))
NP = len(pair_id); pix = lambda u, a: pair_id.get((int(u), int(a)), NP)
tr_p = torch.tensor([pix(u, a) for u, a in zip(tr_u, tr_a)])
te_p = torch.tensor([pix(u, a) for u, a in zip(te_u, te_a)]); NPID = NP + 1
def auc(y, p):
    y = y.numpy(); p = p.detach().numpy(); o = np.argsort(p)
    r = np.empty(len(p)); r[o] = np.arange(len(p))
    pos = (y == 1); npos = pos.sum(); nneg = len(y) - npos
    return float((r[pos].sum() - npos * (npos - 1) / 2) / (npos * nneg)) if npos and nneg else float('nan')
class WD(nn.Module):
    def __init__(s, mode):
        super().__init__(); s.mode = mode
        s.wide = nn.Embedding(NPID, 1); nn.init.zeros_(s.wide.weight)
        s.eu = nn.Embedding(U, EMB); s.ea = nn.Embedding(A, EMB)
        s.mlp = nn.Sequential(nn.Linear(2 * EMB, 16), nn.ReLU(), nn.Linear(16, 1))
        s.b = nn.Parameter(torch.zeros(1))
    def forward(s, u, a, p):
        z = s.b.expand(len(u)).clone()
        if s.mode in ("wide", "both"): z = z + s.wide(p).squeeze(1)
        if s.mode in ("deep", "both"): z = z + s.mlp(torch.cat([s.eu(u), s.ea(a)], 1)).squeeze(1)
        return z
def run(mode):
    torch.manual_seed(0); net = WD(mode)
    opt = torch.optim.Adam(net.parameters(), lr=0.03, weight_decay=3e-4); lf = nn.BCEWithLogitsLoss()
    for _ in range(150):
        opt.zero_grad(); lf(net(tr_u, tr_a, tr_p), tr_y).backward(); opt.step()
    p = torch.sigmoid(net(te_u, te_a, te_p))
    is_exc = torch.tensor([1 if (int(u), int(a)) in exc_set else 0 for u, a in zip(te_u, te_a)])
    unseen = te_p == NP
    return auc(te_y, p), auc(te_y[unseen], p[unseen]), float((p[is_exc == 1] > 0.5).float().mean())
for m in ["wide", "deep", "both"]:
    a, gen, rec = run(m); print(m, round(a, 4), round(gen, 4), round(rec, 3))
# wide  0.7519 0.5039 1.0   deep  0.7532 0.6602 1.0   both  0.7695 0.6726 1.0

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The ablation. You have a working Wide & Deep model. Train two cut-downs &mdash;
            wide-only (drop the deep logit) and deep-only (drop the wide logit) &mdash; on the same toy task,
            same data, same optimizer. On the unseen-pair subset of the test set, which one collapses
            to chance, and why? What does that show?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Keep everything fixed; for wide-only set the model so the logit = bias + wide_logit; for deep-only, logit = bias + deep_logit. — _An honest ablation changes exactly one thing &mdash; which branch is present &mdash; so any difference is attributable to that branch._
- Score the unseen-pair rows (user-app pairs never in training). Wide-only's cross feature has no weight for those pairs (they map to the out-of-vocabulary bucket), so its score is constant there &mdash; AUC near 0.5 (chance). — _A cross-product feature only memorizes pairs it has seen; it cannot rank brand-new pairs._
- Deep-only ranks unseen pairs by embedding similarity, so its unseen-pair AUC is well above chance. — _Embeddings place similar users/apps nearby, letting the deep part generalize to unseen combinations._

**Answer:** Wide-only collapses to chance on unseen pairs (AUC near 0.5): its cross-product
                 feature has no learned weight for a pair it never saw. Deep-only stays well above chance
                 there because its embeddings generalize. This isolates the deep part as the source of
                 generalization &mdash; and, conversely, on the sharp memorized exception pairs the wide part
                 is the strongest. Neither single model wins both, which is the paper's whole point. The
                 CODEVIZ panel shows this split.

</details>

**Problem 2.** You define a cross feature AND(country=US, device=iOS) over one-hot fields. For a user with
            country=US, device=Android, what is $\phi$? For country=US, device=iOS? Walk through Eqn. 1.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Identify the constituents with $c_{ki}=1$: the indicators "country is US" and "device is iOS"; all others have $c_{ki}=0$ and drop out as $x^0=1$. — _Eqn. 1 multiplies only the selected indicators; ignored features contribute a factor of 1._
- country=US, device=Android: "country is US"=1, "device is iOS"=0 &rarr; $\phi = 1\cdot 0 = 0$. — _An AND is 0 if any constituent is 0._
- country=US, device=iOS: both indicators are 1 &rarr; $\phi = 1\cdot 1 = 1$. — _An AND is 1 only when every constituent is present._

**Answer:** For US + Android, $\phi = 0$ (device differs). For US + iOS, $\phi = 1$ (both present).
                 The cross feature fires only for the exact combination, letting the wide part attach a weight
                 to that one rule &mdash; the essence of memorization.

</details>

**Problem 3.** A teammate builds two models &mdash; a wide linear model and a deep network &mdash; trains each
            on its own loss, then at serving time averages their two probabilities. Is this Wide & Deep?
            What is lost?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Compare to Eqn. 3: Wide & Deep sums the two logits inside ONE sigmoid and trains under ONE loss. — _The joint loss couples the branches: the same error term $(p-y)$ flows into both._
- The teammate's setup trains the parts separately and combines only the outputs &mdash; that is an ensemble, which the paper explicitly contrasts with joint training (&sect;3.3). — _Separate losses mean neither part adapts to the other's strengths during training._
- Note the practical loss: in joint training each part can be smaller because it only covers what the other misses; an ensemble must make each part self-sufficient. — _The paper notes the wide part can stay small (a few cross features) when the deep part carries generalization._

**Answer:** No &mdash; averaging two separately trained models is an ensemble, not Wide &
                 Deep. The defining feature is joint training: one loss, one sigmoid over the summed
                 logits, so the gradient couples the two parts and they divide labor. The ensemble loses that
                 coupling and forces each part to be self-sufficient.

</details>